# HSTU · ARGUS  —  Human Simulator

Sequential recommendation architectures trained as a proxy for a language learner.

**Core abstraction.** Each event is `(context, word, action)` where:
- `context` = surrounding sentence tokens (+ exercise class / CEFR level for synthetic data)
- `word` = the token being tested
- `action` = bool — did the user answer correctly?

**Two architecture families (dispatch via `USE_GPU` flag):**

| Mode | Archs | Attention | FFN | Extras |
|---|---|---|---|---|
| CPU / light | `hstu`, `argus` | SiLU-gated / learned bias | GELU | Lightweight |
| GPU / heavy | `hstu_gpu`, `argus_gpu` | SiLU-gated / Flash GQA + RoPE | SwiGLU + sparse MoE | Stochastic depth, grad checkpoint, BF16 AMP |

**Loss (shared across all archs):**
- **BCE** — binary cross-entropy on `action` (click happened or not)
- **MSE** — regression of `sigmoid(logit)` onto `action_prob` (FSRS recall probability from synthetic data; masked/skipped for Duolingo rows which lack a ground-truth probability)
- ARGUS also adds a contrastive **NIP** head (in-batch softmax with hard-negative re-weighting)

**Public API:**
```python
predictor = HSTUPredictor(model)
p   = predictor.predict_action(history, context, word)
ps  = predictor.predict_action_batch(history, [(word, context), ...])
```

**Layout:** 1 Setup · 2 Data · 2.5 Recognition analytics · 3 Models · 4 Training · 5 Eval · 6 API

## 1. Setup and config

In [1]:
# !pip install torch pandas numpy scikit-learn tqdm --quiet
%load_ext autoreload
%autoreload 2

import os, math, json, gzip, urllib.request
from collections import Counter
from dataclasses import dataclass, asdict, field
from pathlib import Path
from typing import Optional, List, Dict, Tuple, Union

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, log_loss
from tqdm.auto import tqdm

torch.manual_seed(0); np.random.seed(0)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_DIR = Path('./data'); DATA_DIR.mkdir(exist_ok=True)
print('PyTorch:', torch.__version__, '| Device:', DEVICE)

PyTorch: 2.12.0+cpu | Device: cpu


/home/arek/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import sys; sys.path.insert(0, str(Path('.').resolve()))

USE_GPU = DEVICE.type == 'cuda'

if USE_GPU:
    from src.models_gpu import (
        ModelConfigGPU, TrainConfigGPU,
        build_model_gpu, train_gpu,
        save_model_gpu, load_model_gpu,
    )
    MODEL_CFG = ModelConfigGPU(arch='hstu_gpu')  # 'hstu_gpu' | 'argus_gpu'
    TRAIN_CFG = TrainConfigGPU(epochs=5)
else:
    from src.models import (
        ModelConfig, TrainConfig,
        build_model, train,
        save_model, load_model,
    )
    MODEL_CFG = ModelConfig(arch='hstu')  # 'hstu' | 'argus'
    TRAIN_CFG = TrainConfig(epochs=5)

from src.models import HSTUPredictor, HistoryEvent, evaluate
from src.slam_loader import load_duolingo_slam
from src.synthetic_dataset import generate_synthetic_events
from src.sequence_dataset import build_dataset, SequenceWindowDataset

from dataclasses import dataclass
from typing import Optional


@dataclass
class DataConfig:
    min_events_per_user:  int = 30
    max_events_per_user:  int = 2000
    min_token_frequency:  int = 5
    val_fraction:         float = 0.1
    subsample_users:      Optional[int] = 5000


DATA_CFG = DataConfig()
print(f'Mode         : {"GPU (heavy)" if USE_GPU else "CPU (light)"}')
print(f'Architecture : {MODEL_CFG.arch.upper()}')
print(f'Device       : {DEVICE}')
if USE_GPU:
    print(f'AMP (BF16)   : {TRAIN_CFG.use_amp and DEVICE.type == "cuda"}')

Mode         : CPU (light)
Architecture : HSTU
Device       : cpu


## 2. Data

**Canonical schema:** `user_id | word | context: list[str] | action: bool | timestamp: int`

Two sources are merged:
- **SLAM** — real Duolingo sentence-level data; context = sentence tokens, exercise = PICK_DEFINITION, no CEFR level.
- **Synthetic** — FSRS-simulated learners; context = `[exercise_class, cefr_level]` for 50 % of users, `[]` for the rest.

See `src/slam_loader.py`, `src/synthetic_dataset.py`, `src/sequence_dataset.py`.

In [3]:
# Load SLAM data  (place en_es.slam.20190204.train under ./data/)
slam_events = load_duolingo_slam(DATA_DIR / 'en_es.slam.20190204.train', max_exercises=500)

# Generate synthetic sessions (equal distribution A1–C2, log-normal session counts)
synthetic_events = generate_synthetic_events(n_humans=20, n_sessions=500, seed=42)

# Merge, build vocab, create windowed train / val datasets
train_ds, val_ds, vocab = build_dataset(slam_events, synthetic_events, DATA_CFG, MODEL_CFG)

print(f'SLAM events:      {len(slam_events):,}   users: {slam_events.user_id.nunique():,}')
print(f'Synthetic events: {len(synthetic_events):,}   users: {synthetic_events.user_id.nunique():,}')
print(f'Train examples:   {len(train_ds):,}')
print(f'Val examples:     {len(val_ds):,}')
print(f'Vocab size:       {vocab["n_tokens"]}')

SLAM events:      1,697   users: 2
Synthetic events: 50,000   users: 20
Train examples:   680
Val examples:     1,300
Vocab size:       1325


## 2.5 Recognition analytics

Как меняется вероятность вспомнить слово (`action_prob`) по мере практики.
`action_prob` — это вероятность правильного ответа, которую рассчитывает FSRS-симулятор в момент упражнения.

- **По словам конкретного пользователя** — кривые роста/забывания по каждому слову
- **Агрегат по всему датасету** — медианная `action_prob` по номеру повторения слова (1-е, 2-е, 3-е …) и успешности

In [4]:
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# ── 1. Кривые recognition для слов одного пользователя ────────────────────────

# Берём пользователей с достаточным числом событий
user_stats = (
    synthetic_events.groupby('user_id')
    .agg(n_events=('action', 'count'), n_words=('word', 'nunique'))
    .query('n_events >= 40 and n_words >= 6')
)
rng_a = np.random.default_rng(7)
sample_users = rng_a.choice(user_stats.index, size=min(3, len(user_stats)), replace=False)

for uid in sample_users:
    user_df = (
        synthetic_events[synthetic_events['user_id'] == uid]
        .sort_values('timestamp')
        .copy()
        .reset_index(drop=True)
    )
    user_df['event_no'] = np.arange(len(user_df))

    # Слова с >= 4 повторениями
    top_words = (
        user_df['word'].value_counts()
        .pipe(lambda s: s[s >= 4])
        .head(10)
        .index.tolist()
    )
    sub = user_df[user_df['word'].isin(top_words)]

    # Маркер: кружок = верно, крестик = неверно
    symbol_map = {True: 'circle', False: 'x'}
    colors = px.colors.qualitative.Plotly

    fig = go.Figure()
    for i, word in enumerate(top_words):
        wdf = sub[sub['word'] == word].sort_values('event_no')
        col = colors[i % len(colors)]
        fig.add_trace(go.Scatter(
            x=wdf['event_no'], y=wdf['action_prob'],
            mode='lines+markers',
            name=word,
            line=dict(color=col, width=1.8),
            marker=dict(
                color=col,
                symbol=[symbol_map[a] for a in wdf['action']],
                size=9,
                line=dict(color='white', width=1),
            ),
            hovertemplate=(
                f'<b>{word}</b><br>'
                'event: %{x}<br>'
                'P(recall): %{y:.3f}<br>'
                '<extra></extra>'
            ),
        ))

    fig.update_layout(
        title=(
            f'Пользователь {uid}: P(recall) каждого слова по событиям<br>'
            '<sup>● = верно  ✕ = неверно</sup>'
        ),
        xaxis_title='Номер события',
        yaxis_title='P(recall) — action_prob',
        yaxis=dict(range=[0, 1]),
        height=420,
        legend=dict(orientation='h', y=-0.25),
    )
    fig.show()

# ── 2. Агрегат: медианная P(recall) по номеру повторения ──────────────────────

# Для каждого (user_id, word) нумеруем попытки
syn_sorted = synthetic_events.sort_values(['user_id', 'word', 'timestamp']).copy()
syn_sorted['rep_no'] = (
    syn_sorted.groupby(['user_id', 'word']).cumcount() + 1
)

# Ограничиваем разумным числом повторений
rep_agg = (
    syn_sorted[syn_sorted['rep_no'] <= 12]
    .groupby(['rep_no', 'action'])['action_prob']
    .agg(['median', 'count'])
    .reset_index()
)
rep_agg.columns = ['rep_no', 'correct', 'median_prob', 'n']

fig2 = go.Figure()
for correct, label, color, dash in [
    (True,  'Верно',   '#00CC96', 'solid'),
    (False, 'Неверно', '#EF553B', 'dot'),
]:
    sub2 = rep_agg[rep_agg['correct'] == correct].sort_values('rep_no')
    fig2.add_trace(go.Scatter(
        x=sub2['rep_no'], y=sub2['median_prob'],
        mode='lines+markers',
        name=label,
        line=dict(color=color, width=2.5, dash=dash),
        marker=dict(size=7),
        hovertemplate='повторение %{x}<br>медиана P(recall): %{y:.3f}<extra></extra>',
    ))

fig2.update_layout(
    title=(
        'Медианная P(recall) по номеру повторения слова<br>'
        '<sup>все синтетические пользователи · разбито по исходу</sup>'
    ),
    xaxis_title='Номер повторения слова',
    yaxis_title='Медианная P(recall)',
    xaxis=dict(dtick=1),
    yaxis=dict(range=[0, 1]),
    height=400,
    legend=dict(title='Исход'),
)
fig2.show()

print(f'Синтетических событий: {len(syn_sorted):,}')
print(rep_agg.pivot(index='rep_no', columns='correct', values='median_prob').rename(
    columns={True: 'верно', False: 'неверно'}).round(3).to_string())

Синтетических событий: 50,000
correct  неверно  верно
rep_no                 
1          0.021  0.284
2          0.207  0.307
3          0.259  0.493
4          0.286  0.590
5          0.299  0.617
6          0.306  0.620
7          0.302  0.653
8          0.295  0.647
9          0.294  0.652
10         0.291  0.644
11         0.291  0.666
12         0.355  0.672


## 3. Models

Both architectures (`hstu`, `argus`) are defined in `src/models.py`.
Switch architecture by changing `ModelConfig(arch=...)` in the config cell.

See `src/models.py` for full implementation details.

In [10]:
if USE_GPU:
    model = build_model_gpu(MODEL_CFG, vocab).to(DEVICE)
else:
    model = build_model(MODEL_CFG, vocab).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
print(f'[{MODEL_CFG.arch.upper()}] {n_params/1e6:.3f}M parameters  |  device: {DEVICE}')

[HSTU] 0.150M parameters  |  device: cpu


## 4. Training loop

**ARGUS** uses a dual-head loss: `FP_loss + nip_weight * NIP_loss`.
The NIP loss is an in-batch sampled softmax over item embeddings.
All other architectures use plain BCE on the recall target.

Training utilities (`argus_loss`, `compute_loss`, `evaluate`, `train`) are in `src/models.py`.

In [11]:
if USE_GPU:
    history = train_gpu(model, train_ds, val_ds, TRAIN_CFG, device=str(DEVICE))
else:
    history = train(model, train_ds, val_ds, TRAIN_CFG, device=str(DEVICE))

results_df = pd.DataFrame(history)
results_df

epoch 1/5:   0%|          | 0/21 [00:00<?, ?it/s]

epoch 1/5: 100%|██████████| 21/21 [00:01<00:00, 11.68it/s, loss=0.7084]


  val  auc=0.7637  ll=0.6129  acc=0.672


epoch 2/5: 100%|██████████| 21/21 [00:01<00:00, 12.29it/s, loss=0.6437]


  val  auc=0.7775  ll=0.5759  acc=0.728


...

epoch 4/5: 100%|██████████| 21/21 [00:01<00:00, 11.07it/s, loss=0.5801]


  val  auc=0.7942  ll=0.5493  acc=0.742


epoch 5/5: 100%|██████████| 21/21 [00:01<00:00, 12.77it/s, loss=0.5314]


  val  auc=0.7934  ll=0.5485  acc=0.736


,auc,log_loss,acc,base_rate,n,epoch,train_loss
0,0.763656,0.612946,0.671538,0.446923,1300,1,0.708394
1,0.777452,0.575919,0.727692,0.446923,1300,2,0.643734
2,0.788330,0.555933,0.739231,0.446923,1300,3,0.608907
3,0.794240,0.549262,0.741538,0.446923,1300,4,0.580143
4,0.793357,0.548512,0.736154,0.446923,1300,5,0.531395


## 5. Evaluation: compare architectures

Run the cell below to train **all three architectures** on the same data split and print a comparison table. Each model is initialized freshly so seeds are comparable.

In [12]:
from dataclasses import asdict

if USE_GPU:
    archs      = ['hstu_gpu', 'argus_gpu']
    _build     = build_model_gpu
    _train     = train_gpu
    _cfg_cls   = ModelConfigGPU
else:
    archs      = ['hstu', 'argus']
    _build     = build_model
    _train     = train
    _cfg_cls   = ModelConfig

comparison = []
for arch in archs:
    torch.manual_seed(0); np.random.seed(0)
    cfg = _cfg_cls(**{**asdict(MODEL_CFG), 'arch': arch})
    m   = _build(cfg, vocab).to(DEVICE)
    print(f'\n=== {arch.upper()} ({sum(p.numel() for p in m.parameters())/1e6:.2f}M params) ===')
    h = _train(m, train_ds, val_ds, TRAIN_CFG, device=str(DEVICE))
    best = max(h, key=lambda r: r.get('auc', 0))
    best['arch'] = arch
    comparison.append(best)

cmp_df = pd.DataFrame(comparison).set_index('arch')[['auc','log_loss','acc','base_rate','train_loss']]
print('\n── Comparison ─────────────────────────────────')
print(cmp_df.to_string())


=== HSTU (0.15M params) ===


epoch 1/5:   0%|          | 0/21 [00:00<?, ?it/s]

epoch 1/5: 100%|██████████| 21/21 [00:01<00:00, 10.86it/s, loss=0.6880]


  val  auc=0.7664  ll=0.6072  acc=0.662


epoch 2/5: 100%|██████████| 21/21 [00:01<00:00, 11.88it/s, loss=0.6229]


...

  val  auc=0.7586  ll=0.5857  acc=0.714


epoch 4/5: 100%|██████████| 21/21 [00:02<00:00,  8.08it/s, loss=1.5553]


  val  auc=0.7685  ll=0.5698  acc=0.725


epoch 5/5: 100%|██████████| 21/21 [00:02<00:00,  8.40it/s, loss=1.3381]


  val  auc=0.7771  ll=0.5631  acc=0.727

── Comparison ─────────────────────────────────
            auc  log_loss       acc  base_rate  train_loss
arch                                                      
hstu   0.798070  0.544182  0.743846   0.446923    0.543332
argus  0.777052  0.563144  0.726923   0.446923    1.338132


In [14]:
# Forgetting-curve sanity check: mean predicted P(correct) per delta bucket.
# A well-trained model should show higher recall for small deltas (recent events).
def forgetting_curve(m, ds, n_batches=50):
    from torch.utils.data import DataLoader
    m.eval()
    loader = DataLoader(ds, batch_size=64, shuffle=True)
    by_bucket = {}
    with torch.no_grad():
        for i, b in enumerate(loader):
            if i >= n_batches: break
            b = {k: v.to(DEVICE) for k, v in b.items()}
            probs = torch.sigmoid(m(b)).cpu().numpy()
            for d, p, y in zip(b['target_delta'].cpu().numpy(), probs,
                                b['target_label'].cpu().numpy()):
                by_bucket.setdefault(int(d), {'pred': [], 'label': []})
                by_bucket[int(d)]['pred'].append(float(p))
                by_bucket[int(d)]['label'].append(float(y))
    import numpy as np
    return pd.DataFrame([{'delta_bucket': d, 'n': len(v['pred']),
                           'mean_pred': np.mean(v['pred']), 'mean_label': np.mean(v['label'])}
                          for d, v in sorted(by_bucket.items())])

forgetting_curve(model, val_ds)

,delta_bucket,n,mean_pred,mean_label
0,0,151,0.895873,0.841060
1,9,6,0.757487,1.000000
2,10,6,0.746789,1.000000
3,11,3,0.806434,0.666667
4,13,1,0.722110,1.000000
5,19,3,0.817529,1.000000
6,21,1126,0.398808,0.387211
7,22,4,0.215918,0.000000


## 6. Save / load + `predict_action` API

`hstu.save` / `hstu.load` work identically for all three architectures.
The saved file includes the `arch` field so the right class is instantiated automatically.

`HSTUPredictor` and `HistoryEvent` are imported from `src/models.py`.

In [9]:
SAVE_PATH = './model.pt'

if USE_GPU:
    save_model_gpu(model, SAVE_PATH)
    reloaded = load_model_gpu(SAVE_PATH, map_location=str(DEVICE))
else:
    save_model(model, SAVE_PATH)
    reloaded = load_model(SAVE_PATH, map_location=str(DEVICE))

print(f'Saved [{MODEL_CFG.arch.upper()}] to {SAVE_PATH} '
      f'({Path(SAVE_PATH).stat().st_size/1e3:.0f} KB)')

predictor = HSTUPredictor(reloaded)

history_events = [
    HistoryEvent('cat',   ['the', 'cat', 'sleeps'],       True,  0),
    HistoryEvent('runs',  ['the', 'dog', 'runs'],          False, 3600),
    HistoryEvent('apple', ['she', 'eats', 'an', 'apple'], True,  7200),
]

p = predictor.predict_action(history_events, ['he', 'reads', 'a', 'book'],
                              'book', now_timestamp=10800)
print(f"\nP(correct on 'book') = {p:.4f}")

candidates = [
    ('book',  ['he', 'reads', 'a', 'book']),
    ('cat',   ['the', 'cat', 'sleeps']),
    ('apple', ['she', 'eats', 'an', 'apple']),
    ('blorg', ['an', 'entirely', 'new', 'sentence']),
]
probs = predictor.predict_action_batch(history_events, candidates, now_timestamp=10800)
print('\nBatch predictions:')
for (w, _), prob in zip(candidates, probs):
    print(f'  {w:10s}  P(correct) = {prob:.4f}')

Saved [HSTU] to ./model.pt (637 KB)

P(correct on 'book') = 0.5914

Batch predictions:
  book        P(correct) = 0.5914
  cat         P(correct) = 0.4307
  apple       P(correct) = 0.5653
  blorg       P(correct) = 0.5259


## Plugging in a new dataset

```python
def load_my_app(path):
    df = pd.read_json(path, lines=True)
    return pd.DataFrame({
        'user_id':   df['user'],
        'word':      df['target_word'],
        'context':   df['sentence_tokens'],   # list[str]; [] is fine
        'action':    df['was_correct'].astype(bool),
        'timestamp': df['ts'].astype('int64'),
        # 'action_prob': df['recall_prob'],   # optional: FSRS-style soft target (float)
        #                                     # omit or set NaN to skip MSE loss component
    })

my_events = load_my_app('events.jsonl')
train_ds, val_ds, vocab = build_dataset(my_events, None, DATA_CFG, MODEL_CFG)
model = build_model(MODEL_CFG, vocab)
```

## Knobs cheatsheet

| Goal | Change |
|---|---|
| Switch to GPU training | `USE_GPU = True` (set before cell-3) or run on CUDA device |
| Switch architecture (CPU) | `ModelConfig(arch='argus')` |
| Switch architecture (GPU) | `ModelConfigGPU(arch='argus_gpu')` |
| Fatter model | `d_model=128, n_heads=4, n_layers=4` |
| More context | `max_context_tokens=32` |
| Longer histories | `max_seq_len=256` |
| ARGUS NIP balance | `argus_nip_weight=0.3` |
| Faster experiments | `DataConfig(subsample_users=500)` |
| More synthetic data | `generate_synthetic_events(n_humans=1000, n_sessions=5000)` |